# Minimal LoRA XLM-R MLP + SVM Ensemble

Small `xlm-roberta-base` notebook in the same style as the other minimal LoRA setups: train XLM-R with LoRA and an MLP classifier head, extract LoRA-tuned CLS embeddings, train a linear SVM on those embeddings, then tune an MLP/SVM posterior ensemble for MAE.

In [ ]:
# Run once on the cluster if needed.
# %pip install -q -U "transformers>=4.40" datasets accelerate "peft>=0.10" scikit-learn joblib

In [ ]:
from pathlib import Path
import inspect
import os
import random
import sys

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import Dataset, Value
from peft import LoraConfig, TaskType, get_peft_model
from sklearn.metrics import accuracy_score, mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.linear_model import SGDClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from transformers import AutoModel, AutoTokenizer, Trainer, TrainingArguments, set_seed

ROOT = Path.cwd()
if not (ROOT / "experiments").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

SEED = 42
MODEL_ID = "xlm-roberta-base"
TRAIN_CSV = ROOT / "data" / "train_lang.csv"
TEST_CSV = ROOT / "data" / "test.csv"
OUTPUT_DIR = ROOT / "outputs" / "minimal_lora_mlp_svm_ensemble_xlmr"

# Use e.g. 20000 for a smoke test; None for full data.
SAMPLE_N = None
VAL_SIZE = 0.10
MAX_LENGTH = 128
N_CLASSES = 5

EPOCHS = 1
BATCH_SIZE = 64
EVAL_BATCH_SIZE = 1024
EMBED_BATCH_SIZE = 256
SVM_TRAIN_N = None  # Set e.g. 100000 for an even faster SVM smoke run.
LR = 1.5e-4
FP16 = torch.cuda.is_available()

LORA_R = 64
LORA_ALPHA = 128
LORA_DROPOUT = 0.05

os.environ.setdefault("WANDB_DISABLED", "true")
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## Data

In [ ]:
df = pd.read_csv(TRAIN_CSV)
df["sentence"] = df["sentence"].fillna("")

if SAMPLE_N is not None and SAMPLE_N < len(df):
    df, _ = train_test_split(df, train_size=SAMPLE_N, random_state=SEED, stratify=df["label"])

train_df, val_df = train_test_split(df, test_size=VAL_SIZE, random_state=SEED, stratify=df["label"])
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

y_train = train_df["label"].to_numpy(dtype=int)
y_val = val_df["label"].to_numpy(dtype=int)

print("train", train_df.shape, "val", val_df.shape)
print(train_df["label"].value_counts().sort_index())

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)


def tokenize(batch):
    out = tokenizer(batch["sentence"], truncation=True, padding="max_length", max_length=MAX_LENGTH)
    out["labels"] = [int(x) for x in batch["label"]]
    return out


def to_dataset(frame):
    ds = Dataset.from_pandas(frame, preserve_index=False)
    ds = ds.map(tokenize, batched=True, remove_columns=ds.column_names)
    ds = ds.cast_column("labels", Value("int64"))
    ds.set_format("torch")
    return ds


train_ds = to_dataset(train_df)
val_ds = to_dataset(val_df)

## LoRA MLP Classifier

In [ ]:
class LoraXLMRMLPClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(MODEL_ID)
        lora_cfg = LoraConfig(
            task_type=TaskType.FEATURE_EXTRACTION,
            r=LORA_R,
            lora_alpha=LORA_ALPHA,
            lora_dropout=LORA_DROPOUT,
            target_modules=["query", "key", "value", "intermediate.dense", "output.dense"],
        )
        self.encoder = get_peft_model(self.encoder, lora_cfg)
        hidden = self.encoder.config.hidden_size
        self.classifier = nn.Sequential(
            nn.Linear(hidden, hidden // 2),
            nn.GELU(),
            nn.Dropout(0.10),
            nn.Linear(hidden // 2, hidden // 4),
            nn.GELU(),
            nn.Dropout(0.05),
            nn.Linear(hidden // 4, N_CLASSES),
        )

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask, **kwargs)
        cls = out.last_hidden_state[:, 0]
        logits = self.classifier(cls)
        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits, labels.long())
        return {"loss": loss, "logits": logits}

    @torch.no_grad()
    def encode(self, input_ids=None, attention_mask=None, **kwargs):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask, **kwargs)
        return F.normalize(out.last_hidden_state[:, 0], p=2, dim=1)


model = LoraXLMRMLPClassifier()
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"trainable params: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")
model.encoder.print_trainable_parameters()

## Metrics

In [ ]:
def softmax_np(logits):
    logits = np.asarray(logits, dtype=np.float64)
    logits = logits - logits.max(axis=1, keepdims=True)
    exp = np.exp(logits)
    return exp / exp.sum(axis=1, keepdims=True)


def expected_mae_risk(probs):
    classes = np.arange(probs.shape[1])
    return np.stack([np.sum(probs * np.abs(pred - classes), axis=1) for pred in classes], axis=1)


def bayes_mae_decode(probs):
    return expected_mae_risk(probs).argmin(axis=1).astype(int)


def evaluate_probs(name, probs, y):
    map_preds = probs.argmax(axis=1).astype(int)
    mae_preds = bayes_mae_decode(probs)
    return {
        "model": name,
        "map_acc": float(accuracy_score(y, map_preds)),
        "map_mae": float(mean_absolute_error(y, map_preds)),
        "bayes_mae": float(mean_absolute_error(y, mae_preds)),
        "counts": np.bincount(mae_preds, minlength=N_CLASSES).tolist(),
    }


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    labels = np.asarray(labels, dtype=int)
    probs = softmax_np(logits)
    map_preds = probs.argmax(axis=1).astype(int)
    mae_preds = bayes_mae_decode(probs)
    return {
        "map_acc": float(accuracy_score(labels, map_preds)),
        "map_mae": float(mean_absolute_error(labels, map_preds)),
        "bayes_mae": float(mean_absolute_error(labels, mae_preds)),
    }


def make_training_args(**kwargs):
    # Transformers renamed evaluation_strategy -> eval_strategy in recent versions.
    params = inspect.signature(TrainingArguments.__init__).parameters
    if "eval_strategy" in params and "evaluation_strategy" in kwargs:
        kwargs["eval_strategy"] = kwargs.pop("evaluation_strategy")
    return TrainingArguments(**kwargs)


def tune_ensemble_weight(mlp_probs, svm_probs, y):
    rows = []
    for w in np.linspace(0, 1, 101):
        probs = w * mlp_probs + (1 - w) * svm_probs
        preds = bayes_mae_decode(probs)
        rows.append({"mlp_weight": float(w), "mae": float(mean_absolute_error(y, preds))})
    scores = pd.DataFrame(rows)
    best = scores.loc[scores["mae"].idxmin()]
    return float(best["mlp_weight"]), scores

## Train LoRA MLP

In [ ]:
args = make_training_args(
    output_dir=str(OUTPUT_DIR / "checkpoints"),
    overwrite_output_dir=True,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    num_train_epochs=EPOCHS,
    evaluation_strategy="steps",
    eval_steps=500,
    logging_steps=100,
    save_strategy="epoch",
    save_total_limit=1,
    fp16=FP16,
    report_to=[],
    remove_unused_columns=False,
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

trainer.train()
metrics = trainer.evaluate()
metrics

## Extract LoRA-Tuned Embeddings

In [ ]:
@torch.no_grad()
def extract_embeddings(ds, batch_size=EMBED_BATCH_SIZE):
    loader = torch.utils.data.DataLoader(ds, batch_size=batch_size)
    device = next(model.parameters()).device
    model.eval()
    chunks = []
    labels = []
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        y = batch.pop("labels", None)
        emb = model.encode(**batch)
        chunks.append(emb.cpu().numpy().astype(np.float32))
        if y is not None:
            labels.append(y.cpu().numpy())
    X = np.vstack(chunks)
    y = np.concatenate(labels) if labels else None
    return X, y


X_train, y_train_check = extract_embeddings(train_ds)
X_val, y_val_check = extract_embeddings(val_ds)
assert np.array_equal(y_train, y_train_check)
assert np.array_equal(y_val, y_val_check)
print(X_train.shape, X_val.shape)

## Train SVM Head

In [ ]:
if SVM_TRAIN_N is not None and SVM_TRAIN_N < len(y_train):
    svm_idx, _ = train_test_split(
        np.arange(len(y_train)),
        train_size=SVM_TRAIN_N,
        random_state=SEED,
        stratify=y_train,
    )
    X_svm = X_train[svm_idx]
    y_svm = y_train[svm_idx]
else:
    X_svm = X_train
    y_svm = y_train

print("SVM train rows:", X_svm.shape[0])

# SGDClassifier with hinge loss is a much faster linear SVM approximation than LinearSVC here.
svm = make_pipeline(
    StandardScaler(),
    SGDClassifier(
        loss="hinge",
        alpha=1e-4,
        penalty="l2",
        class_weight="balanced",
        max_iter=20,
        tol=1e-3,
        early_stopping=True,
        validation_fraction=0.10,
        n_iter_no_change=3,
        average=True,
        n_jobs=-1,
        random_state=SEED,
        verbose=1,
    ),
)
svm.fit(X_svm, y_svm)

## Validation Ensemble

In [ ]:
mlp_logits = trainer.predict(val_ds).predictions
mlp_probs = softmax_np(mlp_logits)
svm_scores = svm.decision_function(X_val)
svm_probs = softmax_np(svm_scores)

best_weight, weight_scores = tune_ensemble_weight(mlp_probs, svm_probs, y_val)
ens_probs = best_weight * mlp_probs + (1 - best_weight) * svm_probs
ens_preds = bayes_mae_decode(ens_probs)

summary = pd.DataFrame([
    evaluate_probs("lora_mlp", mlp_probs, y_val),
    evaluate_probs("lora_embedding_svm", svm_probs, y_val),
    evaluate_probs(f"ensemble_mlp_weight_{best_weight:.2f}", ens_probs, y_val),
])
display(summary)
display(weight_scores.sort_values("mae").head(10))

val_predictions = val_df[["id", "sentence", "label"]].copy()
if "lang" in val_df:
    val_predictions["lang"] = val_df["lang"]
val_predictions["mlp_pred"] = bayes_mae_decode(mlp_probs)
val_predictions["svm_pred"] = bayes_mae_decode(svm_probs)
val_predictions["ensemble_pred"] = ens_preds
val_predictions["ensemble_abs_err"] = np.abs(ens_preds - y_val)
display(pd.crosstab(val_predictions["label"], val_predictions["ensemble_pred"], margins=True))

## Save Artifacts

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(OUTPUT_DIR / "lora_mlp_model"))
tokenizer.save_pretrained(OUTPUT_DIR / "tokenizer")
joblib.dump(svm, OUTPUT_DIR / "svm_head.joblib")
summary.to_csv(OUTPUT_DIR / "validation_summary.csv", index=False)
weight_scores.to_csv(OUTPUT_DIR / "ensemble_weight_scores.csv", index=False)
val_predictions.to_csv(OUTPUT_DIR / "validation_predictions.csv", index=False)
pd.Series({"model_id": MODEL_ID, "best_mlp_weight": best_weight}).to_json(
    OUTPUT_DIR / "ensemble_config.json", indent=2
)
print(OUTPUT_DIR)

## Optional Test Submission

In [ ]:
if TEST_CSV.exists():
    test_df = pd.read_csv(TEST_CSV)
    test_df["sentence"] = test_df["sentence"].fillna("")
    test_ds_raw = Dataset.from_pandas(test_df, preserve_index=False)

    def tokenize_test(batch):
        return tokenizer(batch["sentence"], truncation=True, padding="max_length", max_length=MAX_LENGTH)

    test_ds = test_ds_raw.map(tokenize_test, batched=True, remove_columns=test_ds_raw.column_names)
    test_ds.set_format("torch")

    test_logits = trainer.predict(test_ds).predictions
    test_mlp_probs = softmax_np(test_logits)
    X_test, _ = extract_embeddings(test_ds)
    test_svm_probs = softmax_np(svm.decision_function(X_test))
    test_probs = best_weight * test_mlp_probs + (1 - best_weight) * test_svm_probs
    test_preds = bayes_mae_decode(test_probs)

    submission = pd.DataFrame({"id": test_df["id"], "label": test_preds.astype(int)})
    submission.to_csv(OUTPUT_DIR / "submission.csv", index=False)
    display(submission.head())
    print(OUTPUT_DIR / "submission.csv")
else:
    print("No test CSV found:", TEST_CSV)